# Continuous bandwidth minimizers: ISE, Q₀, Q₁, and Q₁₃/₁₄

For each $n=100,200,\ldots,2000$, compare the sampling distributions of four continuously optimized bandwidths on the common closed interval $h\in[0.01,2]$:

| Selector | Sample criterion |
|---|---|
| ISE | $\int(\widehat f_h-f)^2$ against the known $f=N(0,1)$ |
| $\widehat Q_0$ | $\tfrac12h^2\|\partial_h\widehat f_h\|_2^2$ |
| $\widehat Q_1$ | $\tfrac12\|\partial_h\widehat f_h\|_2^2$ |
| $\widehat Q_{13/14}$ | $\tfrac12h^{1/7}\|\partial_h\widehat f_h\|_2^2$ |

These subscripts use the thesis's **r parameter**, with
$\widehat Q_r(h)=\tfrac12h^{2(1-r)}\|\partial_h\widehat f_h\|_2^2$.
The ISE selector is the sample-specific oracle available here because the simulation density is known. Every criterion is evaluated with analytic Gaussian formulas. The search refines interior candidates continuously and compares both endpoints. Endpoint selections are part of the reported distributions.

The default is **M=1000 paired repetitions at each sample size**. The original 1000-by-1000 sample bank from seed `20260830` is retained exactly. A second block of 1000 observations per repetition is generated independently with `SeedSequence([20260830, 1])` and appended. Thus every sample for n≤1000 is unchanged, and the larger samples extend those same repetitions. Pre-unification results have been removed; new searches use the shared runner. A preview uses the first 12 repetitions of the same complete bank.

Sources: [thesis ISE and `prop:main_thm` defining Q_r](../../../Abschlussarbeit_2021/LaTeX/chapters/5_bandwidth_selection.tex); [exact Gaussian ISE helper](../kde_tools/density.py); [continuous selector implementation and derived log-h derivatives](../kde_tools/continuous_minimizers.py); [earlier paired-sample settings](../inputs/qhat_sampling.json).

Run all cells. A separate resumable checkpoint is saved for each n. The figures are saved automatically as PDFs and also as PNGs. Two pages each show the complete interval and the interior selections with separate bars for the probability at the upper endpoint h=2. All histogram frequencies use M as denominator.


In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

working_directory = Path.cwd().resolve()
code_directory = next(
    (candidate.resolve()
     for root in (working_directory, *working_directory.parents)
     for candidate in (root, root / "Code" / "Code_kde")
     if (candidate / "kde_tools/pdf_export.py").is_file()), None,
)
if code_directory is None:
    raise FileNotFoundError("Run this notebook from the project or Code/Code_kde folder.")
if str(code_directory) not in sys.path:
    sys.path.insert(0, str(code_directory))
os.environ.setdefault("MPLCONFIGDIR", str(code_directory / ".matplotlib-cache"))

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter
from threadpoolctl import threadpool_limits

from kde_tools.continuous_minimizers import (
    CONTINUOUS_SELECTOR_VERSION, METHODS, R_VALUES, continuous_ise_qr_minima,
)
from kde_tools.density import exact_oracle_ise_values
from kde_tools.pdf_export import enable_automatic_pdf_saving

notebook_name = "continuous_ise_qhat_distributions_vs_n"
pdf_output_directory = enable_automatic_pdf_saving(
    notebook_name, output_root=code_directory / "outputs/pdfs"
)
data_output_directory = code_directory / "outputs/data" / notebook_name
image_output_directory = code_directory / "outputs/images" / notebook_name
print(f"Automatic PDF exports: {pdf_output_directory}")

from kde_tools.continuous_runs import CONTINUOUS_RUNNER_VERSION, FIELDS, run_continuous_point
from kde_tools.energy_minimization import energy_search_settings, DEFAULT_ENERGY_SEARCH



## Settings and paired Gaussian samples

The optimization interval is the same [0.01,2] used in the earlier experiment. `HISTOGRAM_POINTS` controls the common display bins only. The minimizers are continuously refined and can fall between bin centers. The three Q criteria are sample energies, optimized separately within each repetition.


In [ ]:
RUN_PROFILE = "thesis"  # "thesis" or "preview"
PROFILES = {"thesis": {"repetitions": 1000}, "preview": {"repetitions": 12}}
SAMPLE_SIZES = [100, 1000, 1500, 2000]
SEED = 20260830
SAMPLE_BANK_REPETITIONS = 1000
SAMPLE_BANK_MAXIMUM_N = 2000
SAMPLE_BANK_BLOCK_SIZE = 1000
H_MIN, H_MAX = 0.01, 2.0
SIGMA2 = 1.0
SCAN_POINTS = DEFAULT_ENERGY_SEARCH.scan_points
SPATIAL_POINTS = DEFAULT_ENERGY_SEARCH.spatial_points
HALF_WIDTH = DEFAULT_ENERGY_SEARCH.half_width
HISTOGRAM_POINTS = 201
WORKERS = 6
REUSE_CHECKPOINTS = True

M = PROFILES[RUN_PROFILE]["repetitions"]
if not 1 <= M <= SAMPLE_BANK_REPETITIONS:
    raise ValueError("Repetitions must fit in the fixed paired sample bank.")
if not SAMPLE_SIZES or min(SAMPLE_SIZES) < 1 or max(SAMPLE_SIZES) > SAMPLE_BANK_MAXIMUM_N:
    raise ValueError("Sample sizes must fit in the fixed sample bank.")
sample_blocks = []
for block_index in range((SAMPLE_BANK_MAXIMUM_N + SAMPLE_BANK_BLOCK_SIZE - 1) // SAMPLE_BANK_BLOCK_SIZE):
    block_seed = SEED if block_index == 0 else np.random.SeedSequence([SEED, block_index])
    sample_blocks.append(np.random.default_rng(block_seed).normal(
        0.0, np.sqrt(SIGMA2), size=(SAMPLE_BANK_REPETITIONS, SAMPLE_BANK_BLOCK_SIZE)
    ))
full_sample_bank = np.concatenate(sample_blocks, axis=1)[:, :SAMPLE_BANK_MAXIMUM_N]
prefix_sample_bank_sha256 = hashlib.sha256(full_sample_bank[:, :SAMPLE_BANK_BLOCK_SIZE].tobytes()).hexdigest()
full_samples = full_sample_bank[:M]
configuration = {
    "version": CONTINUOUS_SELECTOR_VERSION, "energy_search": energy_search_settings(), "runner": CONTINUOUS_RUNNER_VERSION, "methods": list(METHODS),
    "r_values": R_VALUES, "sample_sizes": SAMPLE_SIZES, "repetitions": M,
    "seed": SEED, "sigma2": SIGMA2, "h_min": H_MIN, "h_max": H_MAX,
    "sample_bank_shape": list(full_sample_bank.shape),
    "sample_bank_generation": "fixed_1000_column_blocks_preserving_original_prefix_v1",
    "sample_bank_block_size": SAMPLE_BANK_BLOCK_SIZE,
    "prefix_sample_bank_sha256": prefix_sample_bank_sha256,
    "sample_bank_sha256": hashlib.sha256(full_sample_bank.tobytes()).hexdigest(),
    "scan_points": SCAN_POINTS, "spatial_points": SPATIAL_POINTS,
    "half_width": HALF_WIDTH, "histogram_points": HISTOGRAM_POINTS,
    "source_sha256": {
        filename: hashlib.sha256((code_directory / "kde_tools" / filename).read_bytes()).hexdigest()
        for filename in ("continuous_minimizers.py", "qhat.py")
    },
}
configuration_json = json.dumps(configuration, sort_keys=True)
configuration_id = hashlib.sha256(configuration_json.encode()).hexdigest()[:12]
run_directory = data_output_directory / RUN_PROFILE / configuration_id
image_directory = image_output_directory / RUN_PROFILE / configuration_id
for directory in (run_directory, image_directory):
    directory.mkdir(parents=True, exist_ok=True)
(run_directory / "settings.json").write_text(json.dumps(configuration, indent=2) + "\n")
histogram_bandwidths = np.geomspace(H_MIN, H_MAX, HISTOGRAM_POINTS)
histogram_positions = np.arange(HISTOGRAM_POINTS, dtype=float)
histogram_edges = np.arange(HISTOGRAM_POINTS + 1) - 0.5
continuous_cache_root = code_directory / "outputs/data/shared_continuous_minimizers"
continuous_import_directories = []
print(f"M={M} paired repetitions; n={SAMPLE_SIZES}; continuous interval=[{H_MIN},{H_MAX}]")
print(f"Checkpoint and CSV directory: {run_directory}")


## Compute or resume all four continuous minimizers

The same sample supplies all four selectors. A Fourier scan supplies candidate basins, followed by analytic Gaussian-pair refinement and comparison with both endpoints. A finite scan is a numerical search, rather than a certificate that every stationary point was enumerated. Later validation doubles both scan and Fourier resolution on three repetitions at every n.


In [ ]:
search_settings = {"h_min": H_MIN, "h_max": H_MAX, "sigma2": SIGMA2,
                   "scan_points": SCAN_POINTS, "spatial_points": SPATIAL_POINTS,
                   "half_width": HALF_WIDTH}


def select_one_sample(sample):
    return continuous_ise_qr_minima(sample, **search_settings)


def run_sample_size(n, *, reuse=True):
    shared = run_continuous_point(
        n, full_samples, search_settings, METHODS, run_directory,
        cache_root=continuous_cache_root, configuration=configuration,
        reuse=reuse, workers=WORKERS, import_directories=continuous_import_directories,
    )
    print(f"n={n}: reused {shared['reused_selections']} continuous selections; "
          f"computed {shared['computed_selections']}", flush=True)
    return shared["values"]


results_by_n = {}
with threadpool_limits(limits=1):
    for current_n in SAMPLE_SIZES:
        results_by_n[current_n] = run_sample_size(current_n, reuse=REUSE_CHECKPOINTS)


## Saved distributions, boundary counts, and summaries

Each CSV selection row records a continuously optimized bandwidth and its numerical diagnostics. Histogram frequencies use the same geometric bin centers for all four selectors and divide by M. The summary explicitly counts selections at both endpoints. Interior-only frequencies below keep the same denominator M.

`peak_comparison.csv` records the ISE mode bin, the green interior mode bin, boundary mass, and paired bandwidth comparisons on samples where Q₁₃/₁₄ has an interior minimizer. These are descriptive histogram and sample comparisons.


In [ ]:
frequencies, interior_frequencies, boundary_masses = {}, {}, {}
selection_rows, frequency_rows, summary_rows = [], [], []
for current_n in SAMPLE_SIZES:
    values = results_by_n[current_n]
    frequencies[current_n], interior_frequencies[current_n], boundary_masses[current_n] = {}, {}, {}
    for method_index, method in enumerate(METHODS):
        selected = values[:, method_index, 0]
        boundary = values[:, method_index, 4].astype(int)
        positions = np.interp(np.log(selected), np.log(histogram_bandwidths), histogram_positions)
        counts = np.histogram(positions, bins=histogram_edges)[0]
        interior_counts = np.histogram(positions[boundary != 1], bins=histogram_edges)[0]
        frequencies[current_n][method] = counts / M
        interior_frequencies[current_n][method] = interior_counts / M
        boundary_masses[current_n][method] = float(np.mean(boundary == 1))
        assert counts.sum() == M
        assert interior_counts.sum() + np.sum(boundary == 1) == M
        for repetition in range(M):
            selection_rows.append({
                "n": current_n, "repetition": repetition, "method": method,
                **{field: float(values[repetition, method_index, i]) for i, field in enumerate(FIELDS)},
            })
        for index in range(HISTOGRAM_POINTS):
            frequency_rows.append({
                "n": current_n, "method": method, "grid_index": index,
                "h": float(histogram_bandwidths[index]), "count": int(counts[index]),
                "relative_frequency": float(counts[index] / M),
                "interior_relative_frequency": float(interior_counts[index] / M),
            })
        summary_rows.append({
            "n": current_n, "method": method, "M": M,
            "mean_h": float(np.mean(selected)), "median_h": float(np.median(selected)),
            "sd_h": float(np.std(selected, ddof=1)),
            "q05_h": float(np.quantile(selected, .05)), "q95_h": float(np.quantile(selected, .95)),
            "lower_boundary_count": int(np.sum(boundary == -1)),
            "upper_boundary_count": int(np.sum(boundary == 1)),
            "upper_boundary_frequency": boundary_masses[current_n][method],
            "interior_count": int(np.sum(boundary == 0)),
            "max_interior_candidates": int(np.max(values[:, method_index, 5])),
        })
for filename, rows in (("continuous_minimizers.csv", selection_rows),
                       ("frequencies.csv", frequency_rows), ("summary.csv", summary_rows)):
    with (run_directory / filename).open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
print(" n    selector      median h    at h=2")
for row in summary_rows:
    print(f"{row['n']:4d}  {row['method']:9s}  {row['median_h']:10.5f}  {row['upper_boundary_frequency']:8.1%}")

# Compare the green interior mode with the ISE mode on the common display bins.
peak_comparison_rows = []
for current_n in SAMPLE_SIZES:
    values = results_by_n[current_n]
    ise_mode = float(histogram_bandwidths[np.argmax(frequencies[current_n]["ISE"])])
    green_interior = values[:, METHODS.index("Q13_14"), 4] == 0
    if np.any(green_interior):
        green_mode = float(histogram_bandwidths[np.argmax(interior_frequencies[current_n]["Q13_14"])])
        paired_ise = values[green_interior, METHODS.index("ISE"), 0]
        paired_green = values[green_interior, METHODS.index("Q13_14"), 0]
        mode_ratio = green_mode / ise_mode
        paired_ratio = float(np.median(paired_green / paired_ise))
        paired_difference = float(np.median(paired_green - paired_ise))
    else:
        green_mode = mode_ratio = paired_ratio = paired_difference = None
    peak_comparison_rows.append({
        "n": current_n, "ise_mode_bin_h": ise_mode,
        "q13_14_interior_mode_bin_h": green_mode,
        "interior_mode_ratio_q13_14_over_ise": mode_ratio,
        "q13_14_interior_count": int(np.sum(green_interior)),
        "q13_14_upper_boundary_frequency": boundary_masses[current_n]["Q13_14"],
        "paired_median_h_ratio_on_q13_14_interior_samples": paired_ratio,
        "paired_median_h_difference_on_q13_14_interior_samples": paired_difference,
    })
with (run_directory / "peak_comparison.csv").open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=peak_comparison_rows[0].keys())
    writer.writeheader()
    writer.writerows(peak_comparison_rows)


## Numerical validation

Repeat the continuous searches at doubled scan and Fourier resolution on three paired repetitions at each n. Independently evaluate the exact ISE helper at all four selected bandwidths and both endpoints for these samples; the selected ISE must be no larger than any of those alternatives. Check stationarity and positive curvature for all interior solutions, plus the appropriate one-sided derivative signs at endpoints.


In [ ]:
validation_rows = []
with threadpool_limits(limits=1):
    for current_n in SAMPLE_SIZES:
        values = results_by_n[current_n]
        interior = values[:, :, 4] == 0
        assert np.all(values[:, :, 3][interior] > 0)
        assert np.all(np.abs(values[:, :, 2][interior] / values[:, :, 3][interior]) <= 1e-7)
        assert np.all(values[:, :, 2][values[:, :, 4] == 1] <= 1e-8)
        assert np.all(values[:, :, 2][values[:, :, 4] == -1] >= -1e-8)
        for repetition in sorted({0, M // 2, M - 1}):
            sample = full_samples[repetition, :current_n]
            refined = continuous_ise_qr_minima(
                sample, H_MIN, H_MAX, sigma2=SIGMA2, scan_points=2*SCAN_POINTS-1,
                spatial_points=2*SPATIAL_POINTS, half_width=HALF_WIDTH,
            )
            for method_index, method in enumerate(METHODS):
                difference = abs(refined[method]["h"] - values[repetition, method_index, 0])
                if difference > 1e-7:
                    raise ArithmeticError(f"Refined search changed {method} at n={current_n}.")
                validation_rows.append({"n": current_n, "repetition": repetition, "method": method,
                                        "bandwidth_discrepancy": float(difference)})
            alternative_h = np.r_[values[repetition, :, 0], H_MIN, H_MAX]
            independent_ise = exact_oracle_ise_values(sample, alternative_h, SIGMA2)
            if independent_ise[0] > independent_ise.min() + 1e-10:
                raise ArithmeticError("ISE selector did not minimize the independent candidate scores.")
            if abs(independent_ise[0] - values[repetition, 0, 1]) > 1e-10:
                raise ArithmeticError("Analytic ISE disagrees with the shared exact helper.")
validation_report = {
    "completed_samples": len(SAMPLE_SIZES)*M,
    "continuous_minimizers": len(SAMPLE_SIZES)*M*len(METHODS),
    "maximum_refinement_discrepancy": max(row["bandwidth_discrepancy"] for row in validation_rows),
    "checks": ["interval bounds", "probability totals", "interior stationarity and curvature",
               "one-sided boundary derivative signs", "doubled search resolution",
               "independent exact ISE scores"],
    "refinement_rows": validation_rows,
}
(run_directory / "validation.json").write_text(json.dumps(validation_report, indent=2) + "\n")
print(f"Validated {validation_report['continuous_minimizers']} continuous minimizers.")
print(f"Maximum change after search refinement: {validation_report['maximum_refinement_discrepancy']:.3e}")


## Distribution plots

The first figure shows all selections on the complete bandwidth interval. In the second, each main panel shows the histograms after removing only exact upper-endpoint selections; the inset bars show the probability at h=2. Main-panel frequencies still divide by all M repetitions. The main-panel range includes every occupied interior bin across all sample sizes and methods. Together, the interior histograms and inset bars account for the complete probability distributions.

The twenty sample sizes are split into two readable pages: n=100,…,1000 and n=1100,…,2000. Axis limits are shared across both pages.


In [ ]:
COLORS = {"ISE": "#7B4AB2", "Q0": "#0072B2", "Q1": "#D55E00", "Q13_14": "#009E73"}
LABELS = {"ISE": "ISE", "Q0": r"$\widehat Q_0$", "Q1": r"$\widehat Q_1$",
          "Q13_14": r"$\widehat Q_{13/14}$"}
STYLES = {"ISE": "-", "Q0": "-", "Q1": "--", "Q13_14": "-."}
legend_handles = [Line2D([0], [0], color=COLORS[m], linestyle=STYLES[m], linewidth=2,
                         label=LABELS[m]) for m in METHODS]


def style_panel(ax, current_n, ticks, y_max, right_limit):
    ax.set_title(rf"$n={current_n}$", fontsize=13, pad=8)
    ax.set_xticks(ticks, [f"{histogram_bandwidths[i]:.3g}" for i in ticks])
    ax.set_xlim(-.7, right_limit)
    ax.set_ylim(0, y_max)
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
    ax.set_xlabel("continuously selected bandwidth h", fontsize=10)
    ax.set_ylabel("relative frequency", fontsize=10)
    ax.tick_params(labelsize=9, colors="#304967", labelbottom=True, labelleft=True)
    ax.grid(axis="y", color="#DDE5EC", linewidth=.7)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("#B8C5D1")


def draw_histograms(ax, distributions):
    for method in METHODS:
        ax.stairs(distributions[method], histogram_edges, fill=True,
                  color=COLORS[method], alpha=.07, linewidth=0)
        ax.stairs(distributions[method], histogram_edges, color=COLORS[method],
                  linestyle=STYLES[method], linewidth=1.8 if method == "ISE" else 1.5)


full_y_max = min(1.06, 1.06*max(v.max() for point in frequencies.values() for v in point.values()))
full_ticks = np.unique(np.linspace(0, HISTOGRAM_POINTS-1, 7, dtype=int))

# The four panels included in the thesis, with every interval minimizer.
from kde_tools.thesis_exports import save_minimizer_panel
for current_n in SAMPLE_SIZES:
    panel, panel_ax = plt.subplots(figsize=(4.2, 3.0), layout="constrained")
    draw_histograms(panel_ax, frequencies[current_n])
    style_panel(panel_ax, current_n, full_ticks, full_y_max, HISTOGRAM_POINTS-.3)
    # The thesis caption also reports the mass at the upper endpoint.
    boundary_ax = panel_ax.inset_axes([.56, .48, .28, .42])
    boundary_ax.bar(np.arange(len(METHODS)),
                    [boundary_masses[current_n][m] for m in METHODS],
                    color=[COLORS[m] for m in METHODS], width=.67)
    boundary_ax.set_ylim(0, 1.05)
    boundary_ax.set_yticks([0, .5, 1], ["0%", "50%", "100%"])
    boundary_ax.set_xticks([])
    boundary_ax.tick_params(axis="y", labelsize=8, length=2)
    boundary_ax.set_title(rf"at $h={H_MAX:g}$", fontsize=9, pad=4)
    boundary_ax.patch.set_facecolor("#FAFBFD")
    for side in ("top", "right"):
        boundary_ax.spines[side].set_visible(False)
    panel_ax.set_title(rf"$n={current_n}$; $M={M}$", fontsize=12, pad=8)
    panel.legend(handles=legend_handles, loc="outside lower center", ncol=4, frameon=False, fontsize=8)
    save_minimizer_panel(panel, current_n)
    plt.show()  # Also save automatically under this notebook's PDF directory.
    plt.close(panel)


## Local minima and interval endpoints: discussion control

This fixed-sample control supports the chapter's local-minimum discussion TODO.
It shows detected interior minima and both endpoint comparisons together;
it does not add a second Monte Carlo distribution study.

Sources: [shared search](../kde_tools/energy_minimization.py),
[Gaussian energy](../kde_tools/qhat.py), and [recorded input](../inputs/README.md).


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import sys

KDE_DIR = next(candidate.resolve()
    for ancestor in (Path.cwd(), *Path.cwd().parents)
    for candidate in (ancestor, ancestor / "Code" / "Code_kde")
    if (candidate / "kde_tools/pdf_export.py").is_file())
if str(KDE_DIR) not in sys.path:
    sys.path.insert(0, str(KDE_DIR))
os.environ.setdefault("MPLCONFIGDIR", str(KDE_DIR / ".matplotlib-cache"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from kde_tools import EnergySearchConfig, minimize_qhat, energy_search_settings
from kde_tools.qhat import GaussianQhatObjective
from kde_tools.pdf_export import enable_automatic_pdf_saving

NOTEBOOK_NAME = "continuous_ise_qhat_distributions_vs_n"
pdf_directory = enable_automatic_pdf_saving(NOTEBOOK_NAME)


In [ ]:
# These are the only numerical-search controls needed in this notebook.
SEARCH = EnergySearchConfig()
# Example override: SEARCH = EnergySearchConfig(scan_points=513, scan_backend="exact")
H_MIN, H_MAX = 0.01, 2.0
R_VALUES = (0.0, 13/14, 1.0)

from kde_tools.experiment_inputs import REFERENCE_SAMPLE_PATH, load_reference_sample
sample_path = REFERENCE_SAMPLE_PATH
samples, _ = load_reference_sample()
objective = GaussianQhatObjective(samples)  # Share pair distances and FFT across r values.
analyses = {str(r): minimize_qhat(objective, H_MIN, H_MAX, r=r, config=SEARCH)
            for r in R_VALUES}

summary = pd.DataFrame([dict(
    r=r, local_minima=len(analyses[str(r)]["local_minima"]),
    interval_minimum_h=analyses[str(r)]["global_minimum"]["h"],
    interval_minimum_energy=analyses[str(r)]["global_minimum"]["energy"],
    boundary=analyses[str(r)]["global_minimum"]["boundary"],
    resolution_stable=analyses[str(r)]["diagnostics"]["resolution_stable"])
    for r in R_VALUES])
display(summary)


In [ ]:
# Every local minimum includes `is_global_minimum`, using the configured
# comparison tolerances and all detected candidates plus both endpoints.
local_minima = pd.DataFrame([dict(r=r, **point) for r in R_VALUES
    for point in analyses[str(r)]["local_minima"]])
display(local_minima)
print("Search settings:", energy_search_settings(SEARCH))

settings = dict(search=energy_search_settings(SEARCH), h_min=H_MIN, h_max=H_MAX,
                r_values=R_VALUES, samples_sha256=hashlib.sha256(samples.tobytes()).hexdigest())
run_id = hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()[:12]
output_directory = KDE_DIR / "outputs/data" / NOTEBOOK_NAME / "local_minimum_control" / run_id
output_directory.mkdir(parents=True, exist_ok=True)
(output_directory / "analysis.json").write_text(json.dumps(
    dict(settings=settings, analyses=analyses), indent=2)+"\n")
summary.to_csv(output_directory / "summary.csv", index=False)
local_minima.to_csv(output_directory / "local_minima.csv", index=False)


In [ ]:
h = np.geomspace(H_MIN, H_MAX, 601)
fig, axes = plt.subplots(1, len(R_VALUES), figsize=(13, 4.2), constrained_layout=True)
for ax, r in zip(np.atleast_1d(axes), R_VALUES):
    report = analyses[str(r)]
    ax.loglog(h, [objective.log_energy(np.log(x), r)[0] for x in h], color="#24658a")
    for i, point in enumerate(report["local_minima"]):
        ax.scatter(point["h"], point["energy"], color="#d88321", s=40,
                   label="Interior local minima" if i == 0 else None, zorder=3)
    best = report["global_minimum"]
    ax.scatter(best["h"], best["energy"], color="#744da9", marker="*", s=115,
               label="Interval minimum", zorder=4)
    ax.set(xlabel="Bandwidth h", ylabel="Energy", title=f"r = {r:.6g}")
    ax.legend(fontsize=8)
fig.savefig(output_directory / "energy_minima.png", dpi=160)
plt.show()  # The shared exporter also saves this figure as a PDF.
plt.close(fig)
print("Full reports:", output_directory / "analysis.json")
print("Automatic PDF exports:", pdf_directory)
